# SafeMaint Qwen3.5-9B Colab Server

This notebook starts the same `ai/qwen_service` FastAPI server used by local Docker. It does not mount Google Drive. Use it for development/demo only; company deployment should run the Qwen service on an internal GPU server. This version exposes the server with ngrok.

In [ ]:
# Runtime configuration
SAFE_MAINT_REPO_URL = ""  # optional: https://github.com/your-org/your-repo.git
PROJECT_ROOT = "/content/safemaint"

QWEN_BASE_MODEL = "Qwen/Qwen3.5-9B"
QWEN_LORA_ADAPTER = "/content/qwen_adapter"
QWEN_API_KEY = "change-this-shared-demo-token"
QWEN_LOAD_IN_4BIT = "true"
QWEN_MAX_NEW_TOKENS = "1400"  # structured answer JSON needs more room
QWEN_CLASSIFY_MAX_NEW_TOKENS = "192"

NGROK_AUTH_TOKEN = ""  # required: https://dashboard.ngrok.com/get-started/your-authtoken

# If blank, the notebook will ask you to upload an adapter zip.
ADAPTER_ZIP_URL = ""


In [ ]:
# Install runtime packages. Colab normally already has torch with CUDA.
!pip -q install fastapi uvicorn transformers accelerate peft bitsandbytes safetensors pydantic pyngrok


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

project_root = Path(PROJECT_ROOT)
qwen_service_dir = project_root / "ai" / "qwen_service"
if project_root.exists() and (qwen_service_dir / "main.py").exists():
    print(f"Project already exists: {project_root}")
else:
    if project_root.exists():
        print(f"Removing incomplete project directory: {project_root}")
        shutil.rmtree(project_root)
    if SAFE_MAINT_REPO_URL:
        subprocess.run(["git", "clone", SAFE_MAINT_REPO_URL, str(project_root)], check=True)
    else:
        from google.colab import files
        print("Upload a project zip that contains ai/qwen_service/ or qwen_service/.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No project zip uploaded.")
        archive = Path(next(iter(uploaded.keys()))).resolve()
        unpack_dir = Path("/content/safemaint_upload")
        if unpack_dir.exists():
            shutil.rmtree(unpack_dir)
        shutil.unpack_archive(str(archive), str(unpack_dir))
        # Accept zips created on Windows too. Some zip tools store paths like
        # ai\\qwen_service\\main.py, which Linux treats as flat filenames.
        for path in list(unpack_dir.rglob("*")):
            if "\\" in path.name:
                target = unpack_dir / Path(path.name.replace("\\", "/"))
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(path), str(target))
        candidates = [p for p in unpack_dir.rglob("qwen_service") if (p / "main.py").exists()]
        if not candidates:
            raise RuntimeError("Could not find qwen_service in uploaded zip.")
        candidate = candidates[0]
        if candidate.parent.name == "ai":
            source_root = candidate.parents[1]
            shutil.copytree(source_root, project_root)
        else:
            qwen_service_dir.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(candidate, qwen_service_dir)

if not (qwen_service_dir / "main.py").exists():
    raise RuntimeError(f"qwen_service not found: {qwen_service_dir}")
print(f"Using project root: {project_root}")


In [ ]:
# Prepare LoRA adapter without Google Drive.
from pathlib import Path
import shutil
import subprocess

adapter_dir = Path(QWEN_LORA_ADAPTER)
if adapter_dir.exists() and (adapter_dir / "adapter_config.json").exists():
    print(f"Adapter already exists: {adapter_dir}")
else:
    if adapter_dir.exists():
        shutil.rmtree(adapter_dir)
    adapter_dir.mkdir(parents=True, exist_ok=True)
    if ADAPTER_ZIP_URL:
        zip_path = Path("/content/qwen_adapter.zip")
        subprocess.run(["wget", "-O", str(zip_path), ADAPTER_ZIP_URL], check=True)
    else:
        from google.colab import files
        print("Upload the LoRA adapter zip. It must contain adapter_config.json and adapter_model.safetensors.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No adapter zip uploaded.")
        zip_path = Path(next(iter(uploaded.keys()))).resolve()
    unpack_dir = Path("/content/qwen_adapter_unpacked")
    if unpack_dir.exists():
        shutil.rmtree(unpack_dir)
    shutil.unpack_archive(str(zip_path), str(unpack_dir))
    configs = list(unpack_dir.rglob("adapter_config.json"))
    if not configs:
        raise RuntimeError("adapter_config.json not found in adapter zip.")
    found_adapter_dir = configs[0].parent
    for item in found_adapter_dir.iterdir():
        target = adapter_dir / item.name
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
print(f"Using adapter dir: {adapter_dir}")


In [ ]:
# Start the Qwen FastAPI server.
import os
import subprocess
import time

os.environ["QWEN_BASE_MODEL"] = QWEN_BASE_MODEL
os.environ["QWEN_LORA_ADAPTER"] = QWEN_LORA_ADAPTER
os.environ["QWEN_API_KEY"] = QWEN_API_KEY
os.environ["QWEN_DEVICE"] = "cuda"
os.environ["QWEN_LOAD_IN_4BIT"] = QWEN_LOAD_IN_4BIT
os.environ["QWEN_MAX_NEW_TOKENS"] = QWEN_MAX_NEW_TOKENS
os.environ["QWEN_CLASSIFY_MAX_NEW_TOKENS"] = QWEN_CLASSIFY_MAX_NEW_TOKENS
os.environ.setdefault("HF_HOME", "/content/hf_cache")

subprocess.run("pkill -f 'uvicorn qwen_service.main:app' || true", shell=True, check=False)
subprocess.run("pkill -f 'python.*8020' || true", shell=True, check=False)
subprocess.run("fuser -k 8020/tcp || true", shell=True, check=False)
time.sleep(2)
server_log = open("/content/qwen_server.log", "w")
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "qwen_service.main:app", "--host", "127.0.0.1", "--port", "8020"],
    cwd=str(project_root / "ai"),
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

import urllib.request

def wait_for_local_qwen(timeout_seconds=120):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        if server.poll() is not None:
            break
        try:
            with urllib.request.urlopen("http://127.0.0.1:8020/health/live", timeout=5) as response:
                if response.status == 200:
                    return
        except Exception as exc:
            last_error = exc
        time.sleep(2)
    raise RuntimeError(f"Qwen service did not become healthy on 8020: {last_error}")

wait_for_local_qwen()
print("Qwen service process id:", server.pid)

import json

def check_local_qwen_on(base_url="http://127.0.0.1:8020"):
    """Fail fast unless the local Qwen FastAPI server and model route are really on."""
    status = {
        "server_process_running": server.poll() is None,
        "cuda_requested": os.environ.get("QWEN_DEVICE") == "cuda",
        "health_live": False,
        "openapi_contract_latest": False,
        "model_probe": False,
        "model": None,
        "question_intent": None,
    }
    with urllib.request.urlopen(base_url + "/health/live", timeout=10) as response:
        status["health_live"] = response.status == 200

    with urllib.request.urlopen(base_url + "/openapi.json", timeout=30) as response:
        openapi_payload = json.loads(response.read().decode("utf-8"))
    answer_schema = openapi_payload["components"]["schemas"]["AnswerResponse"]["properties"]
    required_fields = {"answer", "answer_type", "structured_answer", "checklist_items", "used_source_ids", "model"}
    missing = required_fields - set(answer_schema)
    status["openapi_contract_latest"] = not missing
    if missing:
        status["missing_openapi_fields"] = sorted(missing)

    probe_request = urllib.request.Request(
        base_url + "/v1/classify",
        data=json.dumps({"question": "라이트커튼이 무슨 장비야?", "context": {}}, ensure_ascii=False).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {QWEN_API_KEY}",
        },
        method="POST",
    )
    with urllib.request.urlopen(probe_request, timeout=300) as response:
        probe = json.loads(response.read().decode("utf-8"))
    status["model_probe"] = bool(probe.get("analysis") or probe.get("question_intent"))
    status["model"] = probe.get("model")
    status["question_intent"] = (probe.get("analysis") or {}).get("question_intent") or probe.get("question_intent")

    print(json.dumps(status, ensure_ascii=False, indent=2))
    required_ok = status["server_process_running"] and status["health_live"] and status["openapi_contract_latest"] and status["model_probe"]
    if not required_ok:
        raise RuntimeError("Qwen is not fully ON. Check /content/qwen_server.log before sharing the ngrok URL.")
    print("Qwen local server is ON and serving the latest structured API.")
    return status

qwen_local_status = check_local_qwen_on()
!curl -i http://127.0.0.1:8020/health/live
!curl -s http://127.0.0.1:8020/openapi.json | python -m json.tool | head -n 40
!tail -n 40 /content/qwen_server.log


In [ ]:
# Expose the local FastAPI server with ngrok.
if not NGROK_AUTH_TOKEN:
    raise RuntimeError("Set NGROK_AUTH_TOKEN in the first configuration cell.")

from pyngrok import ngrok
import requests
import time

ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(addr=8020, proto="http")
public_url = tunnel.public_url
if public_url.startswith("http://"):
    public_url = public_url.replace("http://", "https://", 1)

print("ngrok Public URL:", public_url)
!curl -i http://127.0.0.1:8020/health/live

headers = {"ngrok-skip-browser-warning": "true"}
time.sleep(3)
health = requests.get(public_url + "/health/live", headers=headers, timeout=30)
print("Public health status:", health.status_code)
print(health.text[:500])
health.raise_for_status()

openapi = requests.get(public_url + "/openapi.json", headers=headers, timeout=30)
print("Public openapi status:", openapi.status_code)
openapi.raise_for_status()
answer_schema = openapi.json()["components"]["schemas"]["AnswerResponse"]["properties"]
required_fields = {"answer", "answer_type", "structured_answer", "checklist_items", "used_source_ids", "model"}
missing = required_fields - set(answer_schema)
assert not missing, f"AnswerResponse schema is missing fields: {sorted(missing)}"

public_probe = requests.post(
    public_url + "/v1/classify",
    json={"question": "라이트커튼이 무슨 장비야?", "context": {}},
    headers={**headers, "Authorization": f"Bearer {QWEN_API_KEY}"},
    timeout=300,
)
print("Public classify status:", public_probe.status_code)
public_probe.raise_for_status()
public_probe_payload = public_probe.json()
assert public_probe_payload.get("analysis") or public_probe_payload.get("question_intent")
print("Public Qwen model probe:", json.dumps({
    "model": public_probe_payload.get("model"),
    "question_intent": (public_probe_payload.get("analysis") or {}).get("question_intent") or public_probe_payload.get("question_intent"),
}, ensure_ascii=False))

print("\nTunnel and OpenAPI schema are reachable.")
print("Public Qwen classify route is ON.")
print("Run the structured smoke test cell next. Share the .env values only after it passes.")


## Structured answer contract smoke test

The uploaded project zip must include the updated `ai/qwen_service/` directory. The classifier now returns a question purpose and the answer endpoint returns both the legacy `answer` string and a validated `structured_answer`. Run the next cell after the model server is ready.

In [ ]:
# Verify the new classify/answer contract before sharing the ngrok URL.
import json
import urllib.request

def post_qwen(path, payload):
    request = urllib.request.Request(
        'http://127.0.0.1:8020' + path,
        data=json.dumps(payload, ensure_ascii=False).encode('utf-8'),
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {QWEN_API_KEY}',
        },
        method='POST',
    )
    with urllib.request.urlopen(request, timeout=300) as response:
        return json.loads(response.read().decode('utf-8'))

classification = post_qwen('/v1/classify', {
    'question': '라이트커튼이 무슨 장비인지 알려줘.',
    'context': {},
})
print(json.dumps(classification, ensure_ascii=False, indent=2))
assert classification['analysis']['question_intent'] == 'component_info'

component_answer = post_qwen('/v1/answer', {
    'question': '라이트커튼이 무슨 장비인지 알려줘.',
    'answer_type': 'component_info',
    'context': {},
    'analysis': classification['analysis'],
    'sources': [{
        'document_id': 'demo-document',
        'chunk_id': 'demo-chunk-1',
        'title': '라이트커튼 제품 설명서',
        'source_type': 'component_manual',
        'excerpt': '라이트커튼은 투광기와 수광기 사이의 광축 차단을 감지하는 안전장치이다.',
    }],
})
print(json.dumps(component_answer, ensure_ascii=False, indent=2))
assert component_answer['answer_type'] == 'component_info'
assert not component_answer['answer'].lstrip().startswith('{')
assert component_answer['structured_answer'] is not None
assert component_answer['structured_answer']['answer_type'] == 'component_info'
assert component_answer['checklist_items'] == []
assert set(component_answer['used_source_ids']) <= {'demo-chunk-1'}
for key in ('main_roles', 'usage_locations', 'precautions'):
    for item in component_answer['structured_answer'][key]:
        assert isinstance(item, dict)
        assert set(item['evidence_chunk_ids']) <= {'demo-chunk-1'}

maintenance_answer = post_qwen('/v1/answer', {
    'question': '라이트커튼 설치 시 주의사항을 알려줘.',
    'answer_type': 'maintenance_guide',
    'context': {},
    'sources': [{
        'document_id': 'demo-document',
        'chunk_id': 'demo-chunk-1',
        'title': '라이트커튼 설치 매뉴얼',
        'source_type': 'component_manual',
        'excerpt': '설치 전 모델별 안전거리와 광축 정렬 기준을 확인한다.',
    }],
})
print(json.dumps(maintenance_answer, ensure_ascii=False, indent=2))
assert maintenance_answer['answer_type'] == 'maintenance_guide'
assert not maintenance_answer['answer'].lstrip().startswith('{')
details = maintenance_answer['structured_answer']
assert details is not None
assert details['answer_type'] == 'maintenance_guide'
assert details['summary']['core_warning']
assert len(details['hazards']) <= 3
for hazard in details['hazards']:
    assert hazard['name']
assert set(maintenance_answer['used_source_ids']) <= {'demo-chunk-1'}
for item in details['manual_steps'] + details['summary']['risk_basis']:
    assert item['evidence_chunk_ids']
    assert set(item['evidence_chunk_ids']) <= {'demo-chunk-1'}
if details['summary']['risk_level'] != '판단 불가':
    assert details['summary']['risk_basis']

print("\nStructured Qwen API smoke test passed.")
print("Set this in each teammate .env:")
print("QWEN_ENABLED=true")
print("QWEN_PROVIDER=colab")
print(f"QWEN_SERVICE_URL={public_url}")
print(f"QWEN_API_KEY={QWEN_API_KEY}")
print("QWEN_TIMEOUT_SECONDS=600")
print("QWEN_ALLOW_COMPANY_CONTEXT=true")
